# BioSkills Lab — Chapter 7
## Biological Data as Matrices and PCA on TCGA
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

In [ ]:
!pip install -q scanpy pandas numpy scikit-learn matplotlib umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
print('Libraries loaded')

## Download TCGA Pan-Cancer Expression Data
We use the TOIL recomputed TCGA dataset hosted on UCSC Xena's public S3 bucket.

In [ ]:
import subprocess
print('Downloading TCGA expression data (~500MB, may take 2-3 min)...')
subprocess.run(['wget', '-q', 'https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/tcga_RSEM_Hugo_norm_count.gz', '-O', 'tcga_expression.tsv.gz'], check=True)
print('Done.')

In [ ]:
expr = pd.read_csv('tcga_expression.tsv.gz', sep='\t', index_col=0)
expr = expr.T
print(f'Samples: {expr.shape[0]}, Genes: {expr.shape[1]}')

In [ ]:
# Data is RSEM norm count - log2 transform
expr_log = np.log2(expr.clip(lower=0) + 1)
print('Log2 transformed')

In [ ]:
print('Downloading labels...')
subprocess.run(['wget', '-q', 'https://toil-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA_phenotype_denseDataOnlyDownload.tsv.gz', '-O', 'tcga_labels.tsv.gz'], check=True)
labels = pd.read_csv('tcga_labels.tsv.gz', sep='\t', index_col=0, compression='gzip')
print(f'Columns: {list(labels.columns[:10])}')

In [ ]:
# Find cancer type column
for col in labels.columns:
    if 'disease' in col.lower() or ('cancer' in col.lower() and 'type' in col.lower()):
        cancer_col = col
        break
print(f'Using: {cancer_col}')

common = expr_log.index.intersection(labels.index)
X_raw = expr_log.loc[common].values
y_raw = labels.loc[common, cancer_col].values
mask = pd.notna(y_raw)
X_raw = X_raw[mask]
y_raw = y_raw[mask]
print(f'Samples: {len(y_raw)}, Cancer types: {len(set(y_raw))}')

In [ ]:
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)
print(f'Classes: {le.classes_}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)
print(f'PCA shape: {X_train_pca.shape}')

In [ ]:
var = pca.explained_variance_ratio_
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.bar(range(1,51), var*100)
plt.xlabel('PC')
plt.ylabel('Variance (%)')
plt.title('Scree Plot')
plt.subplot(1,2,2)
plt.plot(np.cumsum(var)*100)
plt.xlabel('Number of PCs')
plt.ylabel('Cumulative variance (%)')
plt.title('Cumulative Variance')
plt.axhline(80, color='red', linestyle='--', label='80%')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Top 10 PCs explain: {sum(var[:10])*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
for i, ct in enumerate(le.classes_):
    ax.scatter(X_train_pca[y_train==i,0], X_train_pca[y_train==i,1], label=ct, alpha=0.4, s=8)
ax.set_xlabel(f'PC1 ({var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var[1]*100:.1f}%)')
ax.set_title('PCA of TCGA Pan-Cancer Expression')
ax.legend(bbox_to_anchor=(1.05,1), fontsize=6)
plt.tight_layout()
plt.show()